# Person 3 — Portfolio weighting

Person 2 chooses **which stocks** to hold. Your job is to choose **how much money** to put in each selected stock. For each formation month, compare two weighting methods for both benchmarks and each $k\in\{25,50,100\}$: 12 portfolios per month.

At the end of month $t$, use the 60 months $t-59,\ldots,t$, set weights, and hold them during month $t+1$. All returns are decimals (0.01 means 1%). The functions live in `portfolio_weighting.py` so Person 4 can call them from the backtest.

## 1. Inputs and handoff issues

Run `data_extraction.ipynb` or obtain Person 1's two exported CSVs. They are not included in this checkout. Person 2's `cluster_selected_stocks.csv` is included.

- `top500_data/top500_formations.csv`: month, PERMNO, capitalization, coverage flags.
- `top500_data/selected_stock_monthly_history.csv`: monthly stock returns.
- `cluster_selected_stocks.csv`: representatives for each month and k.

**Cluster membership:** Part 2 saves representatives and cluster sizes, but not member PERMNOs. We reconstruct its correlation distance, average linkage, and medoids and verify them against the saved selections. If they differ, stop and obtain memberships from Person 2; do not silently select different stocks.

**Incomplete histories:** Part 2 clusters only current top-500 stocks with 60 observed returns (382 stocks in its December 1999 example). Its clusters therefore do not cover all 500 stocks. Representative weights below redistribute the eligible benchmark mass proportionately to remain fully invested; coverage is reported explicitly. Optimization still targets the full 500-stock benchmark. To represent all 500 directly, the team needs an agreed assignment rule for excluded stocks.

**Benchmark history:** Person 1 exports holdings, but not a benchmark return series. The helper below computes each month's return from the *previous* month's top-500 holdings. Missing constituent returns invalidate the month; no zero filling or survivor reweighting is performed. Person 1 must resolve any such observations and confirm delisting handling in the CRSP extraction before the final backtest.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from portfolio_weighting import (
    prepare_inputs, build_benchmark_returns, reconstruct_clusters,
    representative_weights, weight_one_month,
)

paths = [
    Path("top500_data/top500_formations.csv"),
    Path("top500_data/selected_stock_monthly_history.csv"),
    Path("cluster_selected_stocks.csv"),
]
missing = [str(path) for path in paths if not path.exists()]
if missing:
    raise FileNotFoundError(
        "Run Person 1's notebook or obtain these exports: " + ", ".join(missing)
    )
formations, history, selections = prepare_inputs(*(pd.read_csv(path) for path in paths))
print("Input rows:", len(formations), len(history), len(selections))

## 2. Construct full-universe benchmark targets

For holdings formed at the end of month $t$:

$$b^{VW}_{i,t}=\frac{M_{i,t}}{\sum_{j\in U_t} M_{j,t}},\qquad b^{EW}_{i,t}=\frac{1}{500},\qquad r^B_{t+1}=\sum_{i\in U_t}b_{i,t}r_{i,t+1}.$$

Use the historical, changing benchmark universe in each month of the optimization target. Do not backfill today's constituents to create a different benchmark.

The current export starts in January 1995, so its first computable benchmark return is February 1995. The earliest possible complete 60-month training window ends in **January 2000**, for holding in **February 2000**. To include the original January 2000 holding month, ask Person 1 for December 1994 formation holdings and the necessary return history. December 2025 weights would be for January 2026, whose returns are absent from the current export.

In [ ]:
benchmarks = build_benchmark_returns(formations, history)
display(benchmarks.head())
problem_months = benchmarks.loc[benchmarks["n_missing_returns"].gt(0)]
print("Benchmark months needing Person 1's review:", len(problem_months))
display(problem_months.head(10))

## 3. Representative weights

For each cluster $C_j$, assign its representative the total benchmark mass of the stocks in that cluster, normalized over eligible stocks $E_t$:

$$w^{rep}_{j,t}=\frac{\sum_{i\in C_j}b_{i,t}}{\sum_{i\in E_t}b_{i,t}}.$$

For value weighting, this is cluster capitalization divided by total eligible capitalization. For equal weighting, it is cluster size divided by the number of eligible stocks. Equal-weighted *benchmark* does not mean equal weights on the selected representatives: a representative of 40 stocks should get more weight than one representing 4.

Example: clusters with capitalizations 60, 30, and 10 receive 60%, 30%, and 10% in the value-weighted case. If their sizes are 5, 3, and 2, they receive 50%, 30%, and 20% in the equal-weighted case.

In [ ]:
LOOKBACK = 60
K_VALUES = (25, 50, 100)
# Require all three portfolio sizes and all 60 historical benchmark returns.
selection_sizes = selections.groupby("month")["k"].agg(set)
candidate_months = [
    month for month in sorted(selection_sizes.index)
    if set(K_VALUES).issubset(selection_sizes.loc[month])
    and month + 1 <= history["month"].max()
]
valid_months, excluded_months = [], []
for month in candidate_months:
    window = pd.period_range(month - LOOKBACK + 1, month, freq="M")
    target = benchmarks.reindex(window)[["value_weighted", "equal_weighted"]]
    if np.isfinite(target.to_numpy()).all():
        valid_months.append(month)
    else:
        excluded_months.append({"formation_month": str(month),
                                "reason": "incomplete 60-month benchmark target"})
if not valid_months:
    raise ValueError("No complete benchmark training window. Resolve benchmark coverage with Person 1.")
formation_month = valid_months[0]
print("Example formation:", formation_month, "holding:", formation_month + 1)
print("Excluded candidate months:", len(excluded_months))
display(pd.DataFrame(excluded_months).head(10))
universe, return_matrix, memberships = reconstruct_clusters(
    formation_month, formations, history, selections
)
selected_25 = selections.loc[
    selections["month"].eq(formation_month) & selections["k"].eq(25)
]
w_rep, coverage = representative_weights(
    universe, memberships.loc[memberships["k"].eq(25)], selected_25, "value_weighted"
)
print(f"Eligible share of full benchmark capitalization: {coverage:.2%}")
display(w_rep.sort_values(ascending=False).head(10))

## 4. Tracking-error optimization

Let $X$ be the $60\times k$ matrix of selected stock returns and $y$ the matching 60 historical full-index returns. Active returns are $e=Xw-y$.

The default solves:

$$\min_w\;\frac{1}{59}\sum_{s=1}^{60}(e_s-\bar e)^2
\quad\text{subject to}\quad \sum_{j=1}^{k}w_j=1,\quad 0\leq w_j\leq u.$$

This minimizes **tracking-error variance**; its square root times $\sqrt{12}$ is annualized tracking error. Set `objective="mse"` to minimize $\frac{1}{60}\sum e_s^2$ instead. MSE penalizes average tracking difference as well as variability. Neither objective uses next month's returns. The solver uses [SciPy SLSQP](https://docs.scipy.org/doc/scipy/reference/optimize.minimize-slsqp.html), with an analytic gradient and explicit success/constraint checks.

- `max_weight=1.0`: no additional concentration cap. For a 10% cap use `0.10`. Feasibility requires $ku\geq1$. The cap applies to optimized portfolios; the representative baseline keeps the exact cluster mass rule.
- `ridge=0.0`: pure objective. Positive ridge adds $\lambda\|w-w^{rep}\|_2^2$, which shrinks weights toward the representative baseline. Choose any tuning value using only past information.
- At $k=100$ with only 60 observations, the covariance is singular and the unregularized optimum may be nonunique. The constrained solver does not invert it, but fitted performance can overstate future performance. Ridge can stabilize the fit.
- Some optimal weights may be zero: k is the selected candidate count, not a guarantee of k positive positions. No sector or beta constraints are imposed by default.

In [ ]:
OBJECTIVE = "tracking_error"
MAX_WEIGHT = 1.0
RIDGE = 0.0

weights_example, diagnostics_example = weight_one_month(
    formation_month, formations, history, selections, benchmarks,
    objective=OBJECTIVE, max_weight=MAX_WEIGHT, ridge=RIDGE,
)
display(diagnostics_example[[
    "k", "benchmark", "method", "n_positive_weights", "max_weight",
    "benchmark_mass_covered", "in_sample_te_annualized",
    "in_sample_mean_active_monthly", "in_sample_rmse_monthly",
]])

## 5. Export weights for Person 4

Run the example first, then this full formation loop. Every valid month produces 12 portfolios. Months with incomplete benchmark targets are recorded in `excluded_formation_months.csv`; they must be resolved or explicitly disclosed by the team. Solver failures or mismatched cluster assignments stop execution. They do not silently fall back to different weights.

The diagnostics are **in-sample**, not evidence of out-of-sample tracking quality. Compare all methods on the same holding dates, including disclosure of any excluded dates.

In [ ]:
all_weights, all_diagnostics = [], []
for i, month in enumerate(valid_months):
    weights, diagnostics = weight_one_month(
        month, formations, history, selections, benchmarks,
        objective=OBJECTIVE, max_weight=MAX_WEIGHT, ridge=RIDGE,
    )
    all_weights.append(weights)
    all_diagnostics.append(diagnostics)
    if (i + 1) % 25 == 0 or i + 1 == len(valid_months):
        print(f"Completed {i + 1}/{len(valid_months)} formation months")
portfolio_weights = pd.concat(all_weights, ignore_index=True)
weight_diagnostics = pd.concat(all_diagnostics, ignore_index=True)

keys = ["formation_month", "holding_month", "k", "benchmark", "method"]
checks = portfolio_weights.groupby(keys)["weight"].agg(["sum", "min", "size"])
assert np.allclose(checks["sum"], 1.0, atol=1e-7)
assert checks["min"].ge(0).all()
assert np.array_equal(checks["size"].to_numpy(), checks.index.get_level_values("k").to_numpy())
assert not portfolio_weights.duplicated(keys + ["permno"]).any()

output_dir = Path("part3_output")
output_dir.mkdir(exist_ok=True)
portfolio_weights.to_csv(output_dir / "portfolio_weights.csv", index=False)
weight_diagnostics.to_csv(output_dir / "weight_diagnostics.csv", index=False)
benchmarks.to_csv(output_dir / "benchmark_returns.csv")
pd.DataFrame(excluded_months, columns=["formation_month", "reason"]).to_csv(
    output_dir / "excluded_formation_months.csv", index=False
)
print("Saved Part 3 handoff files to", output_dir)

## 6. What Person 4 does next

`portfolio_weights.csv` contains `formation_month`, `holding_month`, `k`, `benchmark`, `method`, `permno`, `weight`. Join each positive-weight position to `history` using **holding_month and permno**, then calculate $r^P_{t+1}=\sum_jw_{j,t}r_{j,t+1}$. Missing held-stock returns need an explicit data/delisting policy; never silently skip them. Join `benchmark_returns.csv` on holding month and compare to the matching benchmark column.

Report out-of-sample annualized tracking error, return correlation, cumulative returns, and average tracking difference for each of the 12 strategies. The optimizer's in-sample advantage need not survive out of sample.

For turnover at rebalance $t$, first drift the old weights using the just-realized returns:

$$\widetilde w_{i,t}=\frac{w_{i,t-1}(1+r_{i,t})}{1+r^P_t},\qquad
\mathrm{turnover}_t=\frac12\sum_i|w_{i,t}-\widetilde w_{i,t}|.$$

Align PERMNOs over the union of old and new holdings so liquidations count. Decide how to report initial investment, delisting proceeds, and any omitted months. Do not treat separated valid months as adjacent rebalances. Transaction costs and an optional turnover penalty are extensions; this baseline does not optimize turnover.

**Your presentation:** explain the cluster mass rule, show the constrained optimization equation, report coverage and concentration, then let Person 4's future-month results determine which weighting method works best.